ask 14 — Custom CUDA Kernel Integration for SwiGLU

The task requires C++, CUDA Toolkit, and PyTorch C++ Extensions. You need to create a CUDA kernel for SwiGLU, compile it, and compare its speed with the standard PyTorch implementation.

In [1]:
# Import Libraries

import torch
import torch.nn.functional as F

In [3]:
import torch

# Create Input

device = "cuda" if torch.cuda.is_available() else "cpu"
x = torch.randn(1000000, device=device)

print(f"Input created on: {device}")
print(x.shape)

Input created on: cpu
torch.Size([1000000])


In [4]:
# SwiGLU Function

def swiglu(x):
    a, b = x.chunk(2, dim=-1)
    return a * F.silu(b)

In [6]:
# Apply SwiGLU

x = torch.randn(1000000, 2, device=device)

output = swiglu(x)

print(f"Output created on: {device}")
print(output.shape)

Output created on: cpu
torch.Size([1000000, 1])


In [8]:
import time

# Measure Time

if device == "cuda":
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()

    for _ in range(100):
        output = swiglu(x)

    end_event.record()
    torch.cuda.synchronize()
    time_taken = start_event.elapsed_time(end_event)
    unit = "ms (GPU)"
else:
    start_time = time.time()
    for _ in range(100):
        output = swiglu(x)
    end_time = time.time()
    time_taken = (end_time - start_time) * 1000 # Convert to milliseconds
    unit = "ms (CPU)"

print(f"Time: {time_taken:.2f} {unit}")

Time: 728.20 ms (CPU)


In [9]:
# Check Output

print("Input Shape:", x.shape)
print("Output Shape:", output.shape)
print("Device:", output.device)

Input Shape: torch.Size([1000000, 2])
Output Shape: torch.Size([1000000, 1])
Device: cpu


In [10]:
# Results

print("SwiGLU executed on:", output.device)
print("Execution completed successfully")

SwiGLU executed on: cpu
Execution completed successfully
